Get all relevant EKP vitek/phoenix samples

In [ ]:
import pandas as pd
import os

df_vitek = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/VITEK_combined_02_26_interpreted_bin.csv"
)
df_phoenix = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/phoenix_combined_02_26_interpreted_bin.csv"
)


# Remove non PTX samples
df_vitek = df_vitek[df_vitek["Organism_Code"] == "EKP"]
df_phoenix = df_phoenix[df_phoenix["Organism_Code"] == "EKP"]


available_ids_vitek_ = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/VITEK/gff_card_401")
    if f.endswith(".gff")
}
available_ids_phoenix = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/phoenix/gff_card_401")
    if f.endswith(".gff")
}

# Remove unavialible IDs
df_vitek = df_vitek[
    df_vitek["Sample_ID_IfH"].astype(str).isin(available_ids_vitek_)
].reset_index(drop=True)
df_phoenix = df_phoenix[
    df_phoenix["Sample_ID_IfH"].astype(str).isin(available_ids_phoenix)
].reset_index(drop=True)

df_vitek = df_vitek.dropna(subset=df_vitek.columns[2:], how="all")
df_phoenix = df_phoenix.dropna(subset=df_phoenix.columns[2:], how="all")

df_vitek.to_csv("vitek_baseline.csv", index=False)
df_phoenix.to_csv("phoenix_baseline.csv", index=False)

print(len(df_vitek))
print(len(df_phoenix))

869
1606


Clean up 

In [ ]:
# Clean up vitek
df_vitek = pd.read_csv("vitek_baseline.csv")
df_vitek_cleanup = pd.read_csv("samples_used_vitek_15.csv")
df_vitek_cleaned = df_vitek[
    df_vitek["Sample_ID_IfH"].isin(df_vitek_cleanup["Sample_ID_IfH"])
]

df_vitek_cleaned.to_csv("vitek_cleaned.csv", index=False)

# Clean up pheonix
df_phoenix = pd.read_csv("phoenix_baseline.csv")
df_phoenix_cleanup = pd.read_csv("samples_used_phoenix_15.csv")
df_phoenix_cleaned = df_phoenix[
    df_phoenix["Sample_ID_IfH"].isin(df_phoenix_cleanup["Sample_ID_IfH"])
]

df_phoenix_cleaned.to_csv("phoenix_cleaned.csv", index=False)

Random draw multiple times

In [ ]:
import pandas as pd
from pathlib import Path

N_REPEATS = 5
N_TOTAL = 800
N_TRAIN = 600
N_TEST = 200
N_SUB = 300

OUTPUT_DIR = Path("sampling_runs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
vitek_df = pd.read_csv("vitek_cleaned.csv")
phoenix_df = pd.read_csv("phoenix_cleaned.csv")

# Main loop
for repeat in range(N_REPEATS):
    seed = repeat

    run_dir = OUTPUT_DIR / f"run_{repeat}"
    run_dir.mkdir(parents=True, exist_ok=True)

    # Sample 800 per device
    vitek_800 = vitek_df.sample(n=N_TOTAL, random_state=seed)
    phoenix_800 = phoenix_df.sample(n=N_TOTAL, random_state=seed)

    # Split into 600 train / 200 test
    vitek_train = vitek_800.sample(n=N_TRAIN, random_state=seed)
    vitek_test = vitek_800.drop(vitek_train.index)

    phoenix_train = phoenix_800.sample(n=N_TRAIN, random_state=seed)
    phoenix_test = phoenix_800.drop(phoenix_train.index)

    # Mixed subsets (300 each from train)
    vitek_sub = vitek_train.sample(n=N_SUB, random_state=seed)
    phoenix_sub = phoenix_train.sample(n=N_SUB, random_state=seed)

    mixed_train = (
        pd.concat([vitek_sub, phoenix_sub])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    # Save
    vitek_train.to_csv(run_dir / "vitek_train_600.csv", index=False)
    vitek_test.to_csv(run_dir / "vitek_test_200.csv", index=False)

    phoenix_train.to_csv(run_dir / "phoenix_train_600.csv", index=False)
    phoenix_test.to_csv(run_dir / "phoenix_test_200.csv", index=False)

    vitek_sub.to_csv(run_dir / "vitek_train_300.csv", index=False)
    phoenix_sub.to_csv(run_dir / "phoenix_train_300.csv", index=False)

    mixed_train.to_csv(run_dir / "mixed_train_600.csv", index=False)

    print(f"Saved run {repeat} to {run_dir}")

Saved run 0 to sampling_runs/run_0
Saved run 1 to sampling_runs/run_1
Saved run 2 to sampling_runs/run_2
Saved run 3 to sampling_runs/run_3
Saved run 4 to sampling_runs/run_4


Generate DataSet counts

In [1]:
import pandas as pd

vitek_df = pd.read_csv("vitek_cleaned.csv")
phoenix_df = pd.read_csv("phoenix_cleaned.csv")


def count_s_r_per_antibiotic(df: pd.DataFrame) -> pd.DataFrame:
    # Identify antibiotic columns
    antibiotic_cols = [
        col for col in df.columns if col not in ["Sample_ID_IfH", "Organism_Code"]
    ]

    # Count S and R for each antibiotic
    sr_counts = pd.DataFrame(
        {
            "S": (df[antibiotic_cols] == "S").sum(),
            "R": (df[antibiotic_cols] == "R").sum(),
        }
    )

    # Calculate total number of available results
    sr_counts["Total"] = sr_counts["S"] + sr_counts["R"]

    # Remove antibiotics without any S or R result
    sr_counts = sr_counts[sr_counts["Total"] > 0]

    # Convert antibiotic names from index to column
    sr_counts = sr_counts.reset_index()
    sr_counts = sr_counts.rename(columns={"index": "Antibiotic"})

    return sr_counts


vitek_count = count_s_r_per_antibiotic(vitek_df)
print("VITEK")
print(vitek_count)
vitek_count.to_csv("vitek_count.csv", index=False)

phoenix_count = count_s_r_per_antibiotic(phoenix_df)
print("phoenix")
print(phoenix_count)
phoenix_count.to_csv("phoenix_count.csv", index=False)

VITEK
                       Antibiotic    S    R  Total
0                       meropenem  295  553    848
1                       ertapenem  230  580    810
2                      tobramycin  266  426    692
3                     tigecycline   18  665    683
4         piperacillin-tazobactam  226  467    693
5                      cefotaxime   40  672    712
6                     cefpodoxime    0  468    468
7     amoxicillin-clavulanic acid   39  195    234
8                      ampicillin   27  759    786
9                      gentamicin  426  422    848
10                       amikacin  618  230    848
11                    ceftriaxone   16  526    542
12                       cefepime   99  593    692
13                   trimethoprim   25   54     79
14  trimethoprim-sulfamethoxazole    0  751    751
15                  ciprofloxacin   84  764    848
16                    ceftazidime  245  492    737
17               sulfamethoxazole    0   10     10
18                      a